### Delta log P area under the ablation curve

In [25]:
import json
import pandas as pd

# Load results
with open("../results/master_results.json") as f:
    data = json.load(f)
     
# Convert to DataFrame (each experiment = row, metrics = columns)
df = pd.DataFrame(data).T
df.index.name = "experiment"
df = df.reset_index()

# Display as table
df

,experiment,avg_abs_delta,variance_abs_delta,avg_delta,variance_delta,sem_delta,accuracy,num_correct,total,mean_rank,mean_ranking_pct,sem_mean_ranking_pct,n_with_rank,within_top5_pct_count,fraction_within_top5_pct
0,Llama-3.1-8B__Semantic_top0.1,0.172063,0.029386,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Llama-3.1-8B__Semantic_top0.1_random_drop,0.151160,0.026047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Llama-3.1-8B__Semantic_top0.3,0.349702,0.084199,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Llama-3.1-8B__Semantic_top0.3_random_drop,0.334577,0.082518,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Llama-3.2-3B__Semantic_lambada_top0.1,NaN,NaN,-6.770933,18.357257,0.247368,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
5,Llama-3.2-3B__Semantic_lambada_top0.1_random_drop,NaN,NaN,-1.047478,4.036371,0.115994,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
6,Llama-3.2-3B__Semantic_lambada_top0.05,NaN,NaN,-5.050683,18.781683,0.250211,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
7,Llama-3.2-3B__Semantic_lambada_top0.05_random_...,NaN,NaN,-0.715335,3.535874,0.108564,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
8,Llama-3.2-3B__gradient_x_input_lambada_top0.1,NaN,NaN,-6.362518,18.197502,0.246289,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
9,Llama-3.2-3B__Semantic_lambada_top0.2,NaN,NaN,-9.158014,15.859218,0.229922,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# Filter lambada experiments and pivot by drop fraction
lambada_data = {k: v for k, v in data.items() if "lambada" in k.lower()}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_drop" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
    
    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
                        
    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    

    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
lambada_table

method,5%,10%,20%
random,-0.72 ± 0.11,-1.05 ± 0.12,-2.82 ± 0.20
Integrated Grads,-1.05 ± 0.13,-3.24 ± 0.22,-6.56 ± 0.27
Input x Grad,-4.60 ± 0.24,-6.36 ± 0.25,-8.32 ± 0.24
Temperature Scope,-4.95 ± 0.26,-6.82 ± 0.25,-8.92 ± 0.25
Semantic Scope,-5.05 ± 0.25,-6.77 ± 0.25,-9.16 ± 0.23


In [27]:
print(lambada_table.to_markdown())

|                   | 5%           | 10%          | 20%          |
|:------------------|:-------------|:-------------|:-------------|
| random            | -0.72 ± 0.11 | -1.05 ± 0.12 | -2.82 ± 0.20 |
| Integrated Grads  | -1.05 ± 0.13 | -3.24 ± 0.22 | -6.56 ± 0.27 |
| Input x Grad      | -4.60 ± 0.24 | -6.36 ± 0.25 | -8.32 ± 0.24 |
| Temperature Scope | -4.95 ± 0.26 | -6.82 ± 0.25 | -8.92 ± 0.25 |
| Semantic Scope    | -5.05 ± 0.25 | -6.77 ± 0.25 | -9.16 ± 0.23 |


### Most Influential Token

In [29]:

# Filter entries with "loo_rank" (influence ranking vs LOO comparison)
loo_rank_data = {k: v for k, v in data.items() if "loo_rank" in k}

# Map raw method names to display names (same as lambada table)
METHOD_DISPLAY = {
    "Random": "random",
    "Temperature": "Temperature Scope",
    "gradient_x_input": "Input x Grad",
    "Semantic": "Semantic Scope",
    "IG": "Integrated Grads",
    "Fisher": "Fisher Scope",
    
}

# Build lookup: label key → mean_ranking_pct ± SEM string
def fmt_loo_val(entry):
    mean_pct = entry.get("mean_ranking_pct")
    sem_pct = entry.get("sem_mean_ranking_pct")
    if mean_pct is not None and sem_pct is not None:
        return f"{mean_pct:.1f} ± {sem_pct:.1f}%"
    elif mean_pct is not None:
        return f"{mean_pct:.1f}%"
    return None

method_to_val = {}
for label, entry in loo_rank_data.items():
    parts = label.split("__")
    raw_method = parts[1].replace("_lambada_loo_rank", "") if len(parts) >= 2 else label
    display_method = METHOD_DISPLAY.get(raw_method, raw_method)
    method_to_val[display_method] = fmt_loo_val(entry)

# Build table with same row order as lambada: random, Temperature Scope, Semantic Scope, Gradient Input
# row_order = ["random", "Semantic Scope", "Gradient Input", "Temperature Scope"]
row_order = ["random","Integrated Grads", "Semantic Scope", "Input x Grad", "Temperature Scope", "Fisher Scope"]
loo_table = pd.DataFrame(
    {"average ranking": [method_to_val.get(m) for m in row_order]},
    index=row_order,
)
loo_table.index.name = "method"
loo_table

,average ranking
method,
random,51.3 ± 2.9%
Integrated Grads,19.2 ± 2.6%
Semantic Scope,7.0 ± 1.2%
Input x Grad,6.8 ± 1.2%
Temperature Scope,6.5 ± 1.1%
Fisher Scope,5.9 ± 1.1%


In [ ]:
print(loo_table.to_markdown())

| method            | average ranking   |
|:------------------|:------------------|
| random            | 51.3 ± 2.9%       |
| Integrated Grads  | 19.2 ± 2.6%       |
| Semantic Scope    | 7.0 ± 1.2%        |
| Input x Grad      | 6.8 ± 1.2%        |
| Temperature Scope | 6.5 ± 1.1%        |
| Fisher Scope      | 5.9 ± 1.1%        |
